In [0]:
%pip install pypdf==5.1.0 --quiet

In [0]:
# Downloads the CMS PUF data dictionaries (public PDFs), fingerprints them, and stores
# each page's text in a table so the variable descriptions can be parsed reliably.

import hashlib
import io
import os
from datetime import datetime, timezone

import requests
from pypdf import PdfReader

DICTIONARY_DIR = "/Volumes/<your_catalog>/<your_schema>/raw_files/dictionaries/2027"
RAW_TEXT_TABLE = "<your_catalog>.<your_schema>"
REQUEST_HEADERS = {"User-Agent": "qhp-puf-mcp-hackathon/0.1 (public data research)"}

# (dataset, plan_year, official CMS data dictionary URL)
CMS_DICTIONARIES = [
    ("plan_attributes",           2027, "https://www.cms.gov/files/document/planattributes-datadictionary-py27.pdf"),
    ("rates",                     2027, "https://www.cms.gov/files/document/rate-ratadictionary-py27.pdf"),
    ("benefits_and_cost_sharing", 2027, "https://www.cms.gov/files/document/benefitscostsharing-datadictionary-py27.pdf"),
]

os.makedirs(DICTIONARY_DIR, exist_ok=True)
page_rows = []
for dataset, plan_year, url in CMS_DICTIONARIES:
    print(f"Downloading {dataset} dictionary ...")
    try:
        response = requests.get(url, headers=REQUEST_HEADERS, timeout=120)
        response.raise_for_status()
        pdf_bytes = response.content
        pdf_path = f"{DICTIONARY_DIR}/{url.rsplit('/', 1)[-1]}"
        with open(pdf_path, "wb") as f:
            f.write(pdf_bytes)
        pdf_sha256 = hashlib.sha256(pdf_bytes).hexdigest()

        reader = PdfReader(io.BytesIO(pdf_bytes))
        for page_number, page in enumerate(reader.pages, start=1):
            page_rows.append({
                "dataset": dataset, "plan_year": plan_year, "source_url": url,
                "pdf_sha256": pdf_sha256, "page_number": page_number,
                "page_text": page.extract_text() or "",
                "recorded_at": datetime.now(timezone.utc).isoformat(),
            })
        print(f"   done: {len(reader.pages)} pages")
    except Exception as error:
        print(f"   FAILED: {error}")

if page_rows:
    (spark.createDataFrame(page_rows)
        .selectExpr("dataset", "CAST(plan_year AS INT) AS plan_year", "source_url", "pdf_sha256",
                    "CAST(page_number AS INT) AS page_number", "page_text", "recorded_at")
        .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(RAW_TEXT_TABLE))
    print(f"Saved {len(page_rows)} pages to {RAW_TEXT_TABLE}")

In [0]:
# Builds variable_catalog: one row per (dataset, variable, plan_year) the variable exists in.
# Descriptions, labels, and notes come from the CMS PY2027 data dictionary (reused for 2026,
# and labeled so). Anything not found is left empty and listed for review, never invented.

import re
from pyspark.sql import functions as F

SCHEMA = "<your_catalog>.<your_schema>"
CATALOG_TABLE = f"{SCHEMA}.variable_catalog"
DATASETS = ["plan_attributes", "rates"]
DICTIONARY_PLAN_YEAR = 2027
PAGE_HEADER_LINES = 3  # every PDF page starts with: agency name, document title, page number
CMS_DATA_TYPES = {"text", "date", "date/time", "number", "numeric", "decimal", "integer",
                  "currency", "percent", "percentage", "boolean"}
FIELD_LABELS = ("Variable Name:", "Variable Definition:", "Data Type:", "Variable Label:",
                "Allowable Values", "Data Source:", "Field Name from", "Comments:")
NAME_WITH_PREFIX = re.compile(r"^Variable Name:\s*([A-Za-z][A-Za-z0-9]*)$")
NAME_ALONE = re.compile(r"^([A-Za-z][A-Za-z0-9]*)$")
AVAILABILITY = re.compile(r"only (?:available|applicable)", re.IGNORECASE)
EMPTY_COMMENTS = {"", "n/a", "n/a."}
CATALOG_SCHEMA = ("dataset string, variable string, plan_year int, description string, "
                  "cms_variable_name string, variable_label string, cms_data_type string, notes string, "
                  "is_numeric boolean, numeric_column string, unit string, non_null_rows long, "
                  "description_source string")


def to_snake_case(cms_column_name: str) -> str:
    """Same rule the loader used, so catalog names match table column names exactly."""
    name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", cms_column_name.strip())
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_").lower()


def match_key(name: str) -> str:
    """Ignores capitals and punctuation: 'IssuerMarketPlace...' matches 'IssuerMarketplace...'."""
    return re.sub(r"[^a-z0-9]", "", name.lower())


def join_wrapped_lines(lines: list[str]) -> str:
    text = ""
    for line in lines:
        text += line if text.endswith("-") else (" " + line if text else line)
    return " ".join(text.split())


def unique_sentences(text: str) -> str | None:
    """Removes repeated sentences (some PDF entries print a comment twice)."""
    sentences = [s.strip() for s in re.split(r"(?<=\.)\s+", text) if s.strip()]
    result = " ".join(dict.fromkeys(sentences))
    return None if result.lower() in EMPTY_COMMENTS else result


def inline_field(body: list[str], label: str) -> str | None:
    """Reads 'Label: value' plus any wrapped lines, stopping at the next field label."""
    for i, line in enumerate(body):
        if line.startswith(label):
            parts = [line.split(":", 1)[1].strip() if ":" in line else ""]
            for nxt in body[i + 1:]:
                if nxt.startswith(FIELD_LABELS):
                    break
                parts.append(nxt)
            return join_wrapped_lines(parts) or None
    return None


def dictionary_lines(dataset: str) -> list[str]:
    pages = (spark.table(f"{SCHEMA}.dictionary_raw_text")
             .filter((F.col("dataset") == dataset) & (F.col("plan_year") == DICTIONARY_PLAN_YEAR))
             .orderBy("page_number").collect())
    lines = []
    for page in pages:
        lines += [l.strip() for l in page.page_text.splitlines()][PAGE_HEADER_LINES:]
    return [l for l in lines if l]


def locate_entries(lines: list[str], wanted_keys: set[str]) -> dict[str, int]:
    """Finds the line where each variable's entry starts. 'Variable Name: X' lines win;
    a bare 'X' line is used only for entries printed in the labels-first layout."""
    positions = {}
    for pattern in (NAME_WITH_PREFIX, NAME_ALONE):
        for i, line in enumerate(lines):
            m = pattern.match(line)
            if m and match_key(m.group(1)) in wanted_keys and match_key(m.group(1)) not in positions:
                positions[match_key(m.group(1))] = i
    return positions


def parse_entry(entry_lines: list[str]) -> dict:
    body = entry_lines[1:]
    if body and body[0].startswith("Variable Definition:"):  # normal 'Label: value' layout
        comments = inline_field(body, "Comments:")
        return {"definition": inline_field(body, "Variable Definition:"),
                "data_type": inline_field(body, "Data Type:"),
                "label": inline_field(body, "Variable Label:"),
                "notes": unique_sentences(comments) if comments else None}

    # labels-first layout: values follow in order (definition, data type, label, ...)
    definition, data_type, label, label_index = [], None, None, None
    for j, line in enumerate(body[:8]):
        if line.lower() in CMS_DATA_TYPES:
            data_type, label, label_index = line, (body[j + 1] if j + 1 < len(body) else None), j + 1
            break
        if line.startswith(FIELD_LABELS):
            break
        definition.append(line)
    if data_type is None:
        definition = definition[:1]  # layout not recognised: keep only the first line
    notes = None
    if label_index is not None:
        tail = []
        for line in body[label_index + 1:]:
            if line.startswith(FIELD_LABELS):
                break
            tail.append(line)
        kept = [s for s in re.split(r"(?<=\.)\s+", join_wrapped_lines(tail)) if AVAILABILITY.search(s)]
        notes = unique_sentences(" ".join(kept)) if kept else None
    return {"definition": join_wrapped_lines(definition) or None, "data_type": data_type,
            "label": label, "notes": notes}


numeric_info = {(r.dataset, r.variable): r for r in spark.table(f"{SCHEMA}.numeric_profile").collect()}
catalog_rows = []
for dataset in DATASETS:
    manifest = spark.table(f"{SCHEMA}.source_manifest").filter(F.col("dataset") == dataset).collect()
    headers_by_year = {m.plan_year: spark.read.option("header", True).csv(m.csv_file).columns for m in manifest}
    all_cms_names = list(dict.fromkeys(n for cols in headers_by_year.values() for n in cols))

    lines = dictionary_lines(dataset)
    positions = locate_entries(lines, {match_key(n) for n in all_cms_names})
    starts = sorted(positions.values())
    parsed = {}
    for key, start in positions.items():
        end = next((s for s in starts if s > start), len(lines))
        parsed[key] = parse_entry(lines[start:end])

    table = spark.table(f"{SCHEMA}.{dataset}")
    counts = {r.plan_year: r.asDict() for r in table.groupBy("plan_year").agg(
        *[F.count(F.col(to_snake_case(n))).alias(to_snake_case(n)) for n in all_cms_names]).collect()}

    for plan_year, headers in sorted(headers_by_year.items()):
        for cms_name in headers:
            variable = to_snake_case(cms_name)
            info = parsed.get(match_key(cms_name))
            numeric = numeric_info.get((dataset, variable))
            source = "Not found in CMS data dictionary"
            if info and info["definition"]:
                source = f"CMS PY{DICTIONARY_PLAN_YEAR} {dataset} data dictionary"
                if plan_year != DICTIONARY_PLAN_YEAR:
                    source += f" (reused for plan year {plan_year})"
            catalog_rows.append((
                dataset, variable, int(plan_year),
                info["definition"] if info else None, cms_name,
                info["label"] if info else None, info["data_type"] if info else None,
                info["notes"] if info else None,
                numeric is not None, numeric.numeric_column if numeric else None,
                numeric.unit if numeric else None,
                int(counts[plan_year][variable]), source))

catalog = spark.createDataFrame(catalog_rows, CATALOG_SCHEMA)
catalog.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(CATALOG_TABLE)
spark.sql(f"COMMENT ON TABLE {CATALOG_TABLE} IS 'One row per QHP PUF variable per plan year it exists in. "
          f"Descriptions from CMS PY{DICTIONARY_PLAN_YEAR} data dictionaries. Public data.'")
print(f"Saved {catalog.count()} rows to {CATALOG_TABLE}")

display(catalog.groupBy("dataset", "plan_year").agg(
    F.count("*").alias("variables"),
    F.count("description").alias("with_description"),
    F.sum(F.col("is_numeric").cast("int")).alias("numeric_filterable")).orderBy("dataset", "plan_year"))
display(catalog.filter(F.col("description").isNull()).select("dataset", "plan_year", "cms_variable_name"))